# Librerias, autenticación y funciones.

## Librerias

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
import re
import pandas as pd
from IPython.display import display, HTML
import logging

## Autenticación

In [ ]:
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)   # Suprime las advertencias de google_auth_httplib2
auth.authenticate_user()
drive_service = build('drive', 'v3')
sheets_service = build('sheets', 'v4')   # Servicio para Google Sheets

## Funciones

Obtener el ID del **FOLDER**

In [ ]:
def extraer_id_folder(entrada):
    """Extrae el ID de un link de folder o devuelve la entrada si ya es un ID."""
    entrada = entrada.strip()

    # Regex para capturar el ID en URLs de Google Drive
    patron = r"https://drive\.google\.com/drive(?:/u/\d+)?/folders/([a-zA-Z0-9_-]+)"
    coincidencia = re.search(patron, entrada)

    if coincidencia:
        folder_id = coincidencia.group(1)
    else:
        # Si no hay coincidencia, asumimos que la entrada ya es el ID
        folder_id = entrada

    # Llamamos a la visualización bonita
    mostrar_confirmacion_carpeta(folder_id, entrada)

    return folder_id

Visualización estetica para la **FOLDER**

In [ ]:
def mostrar_confirmacion_carpeta(folder_id, entrada_original):
    """Muestra una tarjeta visual con el Nombre Real de la carpeta y su contenido."""

    try:
        # 1. Obtener el nombre de la carpeta
        folder_info = drive_service.files().get(fileId=folder_id, fields="name").execute()
        folder_name = folder_info.get('name', 'Carpeta sin nombre')

        # 2. Obtener el contenido (primeros 5 archivos)
        query = f"'{folder_id}' in parents and trashed = false"
        results = drive_service.files().list(
            q=query,
            pageSize=10,
            fields="files(id, name, mimeType)"
        ).execute()
        items = results.get('files', [])

        lista_archivos = "".join([
            f"<li style='margin-bottom: 4px;'>📄 {item['name'][:40]}{'...' if len(item['name']) > 40 else ''}</li>"
            for item in items[:5]
        ]) if items else "<li style='color: #9aa0a6;'>La carpeta está vacía</li>"

        if len(items) > 5:
            lista_archivos += f"<li style='color: #1a73e8; list-style: none; margin-top:5px;'><i>+ {len(items)-5} archivos más...</i></li>"

    except Exception as e:
        folder_name = "Error de Acceso"
        lista_archivos = f"<li style='color: #d93025;'>No se pudo leer: {str(e)}</li>"

    # 3. Diseño de la Tarjeta Estilizada con el Nombre resaltado
    link_display = (entrada_original[:45] + '...') if len(entrada_original) > 50 else entrada_original

    html_card = f"""
    <div style="font-family: 'Segoe UI', sans-serif; max-width: 650px; margin: 20px 0; border-radius: 16px; overflow: hidden; box-shadow: 0 10px 25px rgba(0,0,0,0.1); background: white; border: 1px solid #e0e0e0;">
        <div style="background: linear-gradient(90deg, #4285F4, #34A853); padding: 20px 25px; display: flex; align-items: center;">
            <span style="font-size: 32px; margin-right: 15px;">📂</span>
            <div>
                <h2 style="margin: 0; color: white; font-size: 1.4em; font-weight: 600; line-height: 1.2;">{folder_name}</h2>
                <span style="color: rgba(255,255,255,0.8); font-size: 0.85em;">Carpeta de Google Drive detectada</span>
            </div>
        </div>

        <div style="padding: 25px;">
            <div style="margin-bottom: 20px;">
                <span style="display: block; font-size: 0.75em; color: #70757a; text-transform: uppercase; font-weight: bold; margin-bottom: 10px;">Archivos encontrados:</span>
                <ul style="margin: 0; padding-left: 20px; color: #3c4043; font-size: 0.95em; line-height: 1.6;">
                    {lista_archivos}
                </ul>
            </div>
        </div>

        <div style="background: #f8f9fa; padding: 12px 25px; border-top: 1px solid #e0e0e0; font-size: 0.8em; color: #9aa0a6; display: flex; justify-content: space-between;">
            <span><b>Origen:</b> {link_display}</span>
            <span style="color: #34A853;">● Conectado</span>
        </div>
    </div>
    """
    display(HTML(html_card))

Obtener el ID de la **HOJA**

In [ ]:
def extraer_id_sheets(entrada):
    """Extrae el ID de un link de Google Sheets o devuelve la entrada si ya es un ID."""
    # Regex para capturar el ID en URLs de Google Sheets
    patron = r"https://docs\.google\.com/spreadsheets/d/([a-zA-Z0-9_-]+)"
    coincidencia = re.search(patron, entrada)
    if coincidencia:
        return coincidencia.group(1)
    # Si no es una URL, asume que ya es un ID directo
    return entrada.strip()

Visualización estetica para la **HOJA**

In [ ]:
def display_fancy_data_card(data_frame_row):
    """Genera una tarjeta HTML estilizada para una sola fila de datos."""
    # Extraemos los valores. Si la columna no existe, ponemos un texto por defecto.
    module_name = data_frame_row['Nombre del Módulo'].values[0] if 'Nombre del Módulo' in data_frame_row.columns else "Módulo no encontrado"
    lesson_name = data_frame_row['Nombre de la lección'].values[0] if 'Nombre de la lección' in data_frame_row.columns else "Lección no encontrada"

    html_content = f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; border-radius: 15px; padding: 25px; margin: 20px 0; background: linear-gradient(145deg, #ffffff, #f0f4f8); border: 1px solid #d1d9e6; box-shadow: 8px 8px 16px #d1d9e6, -8px -8px 16px #ffffff; max-width: 800px;">
        <div style="display: flex; align-items: center; margin-bottom: 20px;">
            <div style="background-color: #4A90E2; color: white; padding: 12px; border-radius: 10px; margin-right: 15px; font-size: 20px;">📘</div>
            <div>
                <span style="display: block; font-size: 0.85em; color: #7f8c8d; text-transform: uppercase; font-weight: bold; letter-spacing: 1px;">Módulo</span>
                <span style="font-size: 1.25em; color: #2c3e50; font-weight: 600;">{module_name}</span>
            </div>
        </div>
        <div style="height: 1px; background: linear-gradient(to right, #4A90E2, transparent); margin: 15px 0;"></div>
        <div style="display: flex; align-items: center;">
            <div style="background-color: #F5A623; color: white; padding: 12px; border-radius: 10px; margin-right: 15px; font-size: 20px;">📝</div>
            <div>
                <span style="display: block; font-size: 0.85em; color: #7f8c8d; text-transform: uppercase; font-weight: bold; letter-spacing: 1px;">Lección</span>
                <span style="font-size: 1.2em; color: #34495e;">{lesson_name}</span>
            </div>
        </div>
    </div>
    """
    return HTML(html_content)

Leer datos de la **HOJA**

In [ ]:
def visualizar_hoja_pandas(spreadsheet_id, nombre_hoja=None):
    # 1. Obtener los datos de la hoja (esta es la parte que faltaba)
    rango = f"{nombre_hoja}!A1:Z" if nombre_hoja else "A1:Z"
    try:
        result = sheets_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id, range=rango).execute()
        values = result.get('values', [])

        if not values:
            print('No se encontraron datos.')
            return pd.DataFrame()

        # 2. Crear el DataFrame
        header = values[0]
        data = values[1:]
        df_completo = pd.DataFrame(data, columns=header)

        # 3. Filtrar solo las columnas que nos interesan para la tarjeta
        columnas_target = ['Nombre del Módulo', 'Nombre de la lección']
        columnas_existentes = [col for col in columnas_target if col in df_completo.columns]
        df_final = df_completo[columnas_existentes]

        # 4. Mostrar la tarjeta estética
        if not df_final.empty:
            display(display_fancy_data_card(df_final.head(1)))

        return df_final

    except Exception as e:
        print(f"Error al acceder a la hoja: {e}")
        return pd.DataFrame()

# Identificadores de hoja y carpeta de imágenes

Identificador de Sheets

In [ ]:
# Solicitar al usuario el ID o link de Google Sheets
SPREADSHEET_LINK = "https://docs.google.com/spreadsheets/d/1mQ9kFUft1SrSJs1l0IkZcphLQ8KtxX96NiDrxfx4MdI/edit?usp=sharing" #@param {type:"string"}
SHEET_NAME = "L1" #@param {type:"string"}

# Procesar el ID
SPREADSHEET_ID = extraer_id_sheets(SPREADSHEET_LINK.strip())

if not SPREADSHEET_ID:
    print("No se pudo extraer un ID válido. Verifica el link o ID ingresado.")
else:
    # Si no se escribió nombre de hoja, tomar la primera
    if SHEET_NAME.strip() == "":
        spreadsheet_metadata = sheets_service.spreadsheets().get(
            spreadsheetId=SPREADSHEET_ID
        ).execute()
        sheets = spreadsheet_metadata.get('sheets', [])
        SHEET_NAME = sheets[0].get('properties', {}).get('title')
        print(f"Hoja seleccionada automáticamente: {SHEET_NAME}")

    df_hoja = visualizar_hoja_pandas(SPREADSHEET_ID, SHEET_NAME)


Error al acceder a la hoja: 21 columns passed, passed data had 16 columns


In [ ]:
# @title Carpeta de imágenes para simuladores.
Link_Carpeta = "https://drive.google.com/drive/folders/1r9BrdlEZDpiaomWid5azz0c4htLzZo6H?usp=drive_link" #@param {type:"string"}
Tipo_Recurso = "simulador" #@param {type:"string"}
Nombre_Recurso = Tipo_Recurso

FOLDER_ID_Simuladores = extraer_id_folder(Link_Carpeta)

In [ ]:
# @title Carpeta de imágenes para libros.
Link_Carpeta = "https://drive.google.com/drive/folders/1_YUinixXqD-imnFJ09e-R5Q7PCop3JrQ?usp=drive_link" #@param {type:"string"}
Tipo_Recurso= "libro" #@param {type:"string"}
Nombre_Recurso2 = Tipo_Recurso

FOLDER_ID_Libros = extraer_id_folder(Link_Carpeta)

# 0. Imágenes de los videos de introducción

In [ ]:
# 1. Función para convertir índice de columna a letra
def col_index_to_letter(index):
    letter = ''
    while index >= 0:
        letter = chr(index % 26 + 65) + letter
        index = index // 26 - 1
    return letter

# 2. Función mejorada para identificar el tipo de video y obtener su miniatura
def get_video_thumbnail(url):
    if not isinstance(url, str) or not url:
        return ""

    # --- Lógica para YouTube ---
    yt_patterns = [
        r'(?:youtube\.com\/watch\?v=|youtu\.be\/|youtube\.com\/embed\/|youtube\.com\/v\/|youtube\.com\/shorts\/)([a-zA-Z0-9_-]{11})',
        r'^([a-zA-Z0-9_-]{11})$'
    ]
    for pattern in yt_patterns:
        match = re.search(pattern, url)
        if match:
            video_id = match.group(1)
            return f"https://img.youtube.com/vi/{video_id}/maxresdefault.jpg"

    # --- Lógica para JoVE ---
    # Detecta tanto "jove.com" directo como proxies de biblioteca ("jove-com")
    if re.search(r'jove[.\-]com', url):
        jove_match = re.search(r'[?&]id=(\d+)', url)
        if jove_match:
            jove_id = jove_match.group(1)
            return f"https://cloudfront.jove.com/files/media/science-education/science-education-thumbs/{jove_id}.jpg"

    return "" # Si no es ninguno, devuelve vacío

# --- PROCESO PRINCIPAL ---

# Obtener todos los datos de la hoja
result = sheets_service.spreadsheets().values().get(
    spreadsheetId=SPREADSHEET_ID,
    range=f"{SHEET_NAME}!A1:Z"
).execute()
values = result.get('values', [])

if not values:
    print("No se encontraron datos en la hoja.")
else:
    headers = values[0]
    source_col = None   # "Link del video"
    target_col = None   # "Link de la Imagen del video"

    for i, h in enumerate(headers):
        if h == "Link del video de Introduccion":
            source_col = i
        if h == "Link de la Imagen del video de introduccion":
            target_col = i

    if source_col is None or target_col is None:
        print("Error: Asegúrate de que las columnas 'Link del video de Introduccion' y 'Link de la Imagen del video de introduccion' existan.")
    else:
        # 3. Procesar las filas y generar la lista de miniaturas
        thumbnail_urls = []
        # Empezamos desde la fila 1 (índice 2)
        for row_idx in range(1, len(values)):
            row = values[row_idx]

            # Identificamos el valor de la celda en la fila actual
            video_url = row[source_col] if source_col < len(row) else None

            # Obtenemos la miniatura procesando el link (YouTube o JoVE)
            thumbnail_url = get_video_thumbnail(video_url)
            thumbnail_urls.append([thumbnail_url])

        # Determinar rango de escritura exacto para que quede "al lado" en la fila correcta
        start_row = 2
        end_row = start_row + len(thumbnail_urls) - 1
        target_letter = col_index_to_letter(target_col)
        update_range = f"{SHEET_NAME}!{target_letter}{start_row}:{target_letter}{end_row}"

        # Actualizar la columna destino
        body = {'values': thumbnail_urls}
        sheets_service.spreadsheets().values().update(
            spreadsheetId=SPREADSHEET_ID,
            range=update_range,
            valueInputOption='RAW',
            body=body
        ).execute()

        print(f"Proceso completado. Se analizaron {len(thumbnail_urls)} filas.")

Error: Asegúrate de que las columnas 'Link del video de Introduccion' y 'Link de la Imagen del video de introduccion' existan.


#1. Imágenes de Videos

In [ ]:
# 1. Función para convertir índice de columna a letra
def col_index_to_letter(index):
    letter = ''
    while index >= 0:
        letter = chr(index % 26 + 65) + letter
        index = index // 26 - 1
    return letter

# 2. Función mejorada para identificar el tipo de video y obtener su miniatura
def get_video_thumbnail(url):
    if not isinstance(url, str) or not url:
        return ""

    # --- Lógica para YouTube ---
    yt_patterns = [
        r'(?:youtube\.com\/watch\?v=|youtu\.be\/|youtube\.com\/embed\/|youtube\.com\/v\/|youtube\.com\/shorts\/)([a-zA-Z0-9_-]{11})',
        r'^([a-zA-Z0-9_-]{11})$'
    ]
    for pattern in yt_patterns:
        match = re.search(pattern, url)
        if match:
            video_id = match.group(1)
            return f"https://img.youtube.com/vi/{video_id}/maxresdefault.jpg"

    # --- Lógica para JoVE ---
    # Detecta tanto "jove.com" directo como proxies de biblioteca ("jove-com")
    if re.search(r'jove[.\-]com', url):
        jove_match = re.search(r'[?&]id=(\d+)', url)
        if jove_match:
            jove_id = jove_match.group(1)
            return f"https://cloudfront.jove.com/files/media/science-education/science-education-thumbs/{jove_id}.jpg"

    return "" # Si no es ninguno, devuelve vacío

# --- PROCESO PRINCIPAL ---

# Obtener todos los datos de la hoja
result = sheets_service.spreadsheets().values().get(
    spreadsheetId=SPREADSHEET_ID,
    range=f"{SHEET_NAME}!A1:Z"
).execute()
values = result.get('values', [])

if not values:
    print("No se encontraron datos en la hoja.")
else:
    headers = values[0]
    source_col = None   # "Link del video"
    target_col = None   # "Link de la Imagen del video"

    for i, h in enumerate(headers):
        if h == "Link del video":
            source_col = i
        if h == "Link de la Imagen del video":
            target_col = i

    if source_col is None or target_col is None:
        print("Error: Asegúrate de que las columnas 'Link del video' y 'Link de la Imagen del video' existan.")
    else:
        # 3. Procesar las filas y generar la lista de miniaturas
        thumbnail_urls = []
        # Empezamos desde la fila 1 (índice 2)
        for row_idx in range(1, len(values)):
            row = values[row_idx]

            # Identificamos el valor de la celda en la fila actual
            video_url = row[source_col] if source_col < len(row) else None

            # Obtenemos la miniatura procesando el link (YouTube o JoVE)
            thumbnail_url = get_video_thumbnail(video_url)
            thumbnail_urls.append([thumbnail_url])

        # Determinar rango de escritura exacto para que quede "al lado" en la fila correcta
        start_row = 2
        end_row = start_row + len(thumbnail_urls) - 1
        target_letter = col_index_to_letter(target_col)
        update_range = f"{SHEET_NAME}!{target_letter}{start_row}:{target_letter}{end_row}"

        # Actualizar la columna destino
        body = {'values': thumbnail_urls}
        sheets_service.spreadsheets().values().update(
            spreadsheetId=SPREADSHEET_ID,
            range=update_range,
            valueInputOption='RAW',
            body=body
        ).execute()

        print(f"Proceso completado. Se analizaron {len(thumbnail_urls)} filas.")

Proceso completado. Se analizaron 7 filas.


# 2. Imágenes simuladores o libros

In [ ]:
def extraer_numero(nombre):
    """Extrae el primer número encontrado en el nombre del archivo.
    Retorna float('inf') si no encuentra ninguno."""
    coincidencia = re.search(r'\d+', nombre)
    return int(coincidencia.group()) if coincidencia else float('inf')

def obtener_columna_por_encabezado(spreadsheet_id, sheet_name, encabezado):
    """Lee la fila 1 y retorna la letra de columna que coincide con el encabezado dado."""
    range_fila1 = f"'{sheet_name}'!1:1"
    result = sheets_service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=range_fila1
    ).execute()
    valores = result.get('values', [[]])[0]  # Lista de celdas en la fila 1

    for idx, celda in enumerate(valores):
        if celda.strip() == encabezado.strip():
            # Convertir índice (0-based) a letra de columna (A, B, ..., Z, AA, AB, ...)
            col_letra = indice_a_letra(idx)
            print(f"Columna '{encabezado}' encontrada en: {col_letra} (índice {idx})")
            return col_letra

    raise ValueError(f"No se encontró el encabezado '{encabezado}' en la fila 1.")


def indice_a_letra(indice):
    """Convierte un índice 0-based a letra de columna de Sheets (0→A, 25→Z, 26→AA, ...)."""
    letra = ''
    while indice >= 0:
        letra = chr(indice % 26 + ord('A')) + letra
        indice = indice // 26 - 1
    return letra


def buscar_fila(spreadsheet_id, sheet_name, col_letra):
    """Lee la columna 'Título simulador' y retorna el número de fila del primer dato."""
    range_col = f"'{sheet_name}'!{col_letra}:{col_letra}"
    result = sheets_service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=range_col
    ).execute()
    valores = result.get('values', [])

    for row_idx, celda in enumerate(valores):
        if row_idx == 0:
            continue  # Saltar encabezado (fila 1)
        if celda and celda[0].strip():
            print(f"Primera fila con dato en '{col_letra}': fila {row_idx + 1}")
            return row_idx + 1  # Convertir índice 0-based a número de fila Sheets

    return 2  # Fallback si la columna está vacía


def obtener_enlaces_directos(folder_id, spreadsheet_id, sheet_name, tipo):
    # 1. Buscar columnas por encabezado en fila 1
    col_titulo = obtener_columna_por_encabezado(spreadsheet_id, sheet_name, f"Link del {tipo}")
    col_dest   = obtener_columna_por_encabezado(spreadsheet_id, sheet_name, f"Link de la imagen del {tipo}")

    # 2. Obtener imágenes
    query = f"'{folder_id}' in parents and mimeType contains 'image/' and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, name)", pageSize=1000).execute()
    items = results.get('files', [])
    if not items:
        print('No se encontraron imágenes.')
        return None

    # 3. Ordenar por número
    items_ordenados = sorted(items, key=lambda x: extraer_numero(x['name']))
    print(f"\nSe encontraron {len(items_ordenados)} imágenes.\n")

    # 4. Contar blancos en col_ref para calcular fila de inicio
    fila_inicio = buscar_fila(spreadsheet_id, sheet_name, col_titulo)
    print(f"Los enlaces se escribirán desde {col_dest}{fila_inicio}\n")

    # 5. Limpiar columna destino
    try:
        sheets_service.spreadsheets().values().clear(
            spreadsheetId=spreadsheet_id,
            range=f"'{sheet_name}'!{col_dest}2:{col_dest}"
        ).execute()
        print(f"Columna '{col_dest}' limpiada correctamente.")
    except Exception as e:
        print(f"No se pudo limpiar la columna destino: {e}")

    # 6. Recolectar enlaces
    datos = {}
    enlaces = []
    for i, item in enumerate(items_ordenados, start=1):
        file_id   = item['id']
        file_name = item['name']
        direct_link = f"https://drive.google.com/thumbnail?id={file_id}&sz=w1000"

        numero = extraer_numero(file_name)
        clave  = numero if numero != float('inf') else i

        datos[clave] = {'nombre': file_name, 'enlace': direct_link}
        enlaces.append([direct_link])

        print(f"Imagen {clave} (orden {i}):")
        print(f"  Nombre: {file_name}")
        print(f'  Enlace: <img src="{direct_link}">\n')

    # 7. Escribir desde la fila calculada
    if enlaces:
        range_escritura = f"'{sheet_name}'!{col_dest}{fila_inicio}"
        sheets_service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=range_escritura,
            valueInputOption='RAW',
            body={'values': enlaces}
        ).execute()
        print(f"Se escribieron {len(enlaces)} enlaces desde {col_dest}{fila_inicio}.")

    return datos


# Ejecutar Simuladores
resultados_Simuladores = obtener_enlaces_directos(FOLDER_ID_Simuladores, SPREADSHEET_ID, SHEET_NAME, Nombre_Recurso)

# Ejecutar Libros
resultados_Libros = obtener_enlaces_directos(FOLDER_ID_Libros, SPREADSHEET_ID, SHEET_NAME, Nombre_Recurso2)

print("\n--- Diccionario completo ---")
print(resultados_Simuladores)
print("\n\n\n--- Diccionario completo ---")
print(resultados_Libros)

Columna 'Link del simulador' encontrada en: L (índice 11)
Columna 'Link de la imagen del simulador' encontrada en: M (índice 12)

Se encontraron 4 imágenes.

Primera fila con dato en 'L': fila 13
Los enlaces se escribirán desde M13

Columna 'M' limpiada correctamente.
Imagen 1 (orden 1):
  Nombre: 1.png
  Enlace: <img src="https://drive.google.com/thumbnail?id=11zoe7dAMXPZlVtwOnbXnQuQ1yGU-lN1u&sz=w1000">

Imagen 2 (orden 2):
  Nombre: 2.png
  Enlace: <img src="https://drive.google.com/thumbnail?id=11tiIxKQUGnL1lZvrv6zzeTVHAhw0mhXn&sz=w1000">

Imagen 3 (orden 3):
  Nombre: 3.png
  Enlace: <img src="https://drive.google.com/thumbnail?id=10wTwaBX5IBvFDo1600pK1eCHLXV4OCa_&sz=w1000">

Imagen 4 (orden 4):
  Nombre: 4.png
  Enlace: <img src="https://drive.google.com/thumbnail?id=1a9GrNElxBXW3wJUpUrF_CW1hdCnw0r9z&sz=w1000">

Se escribieron 4 enlaces desde M13.
Columna 'Link del libro' encontrada en: O (índice 14)
Columna 'Link de la imagen del libro' encontrada en: P (índice 15)

Se encontraro

# Contacto

Héctor Franco → 3158118672